# 6.390 Spring 2024 Homework 6

**If you haven't already, please hit :**

`File` -> `Save a Copy in Drive`

**to copy this notebook to your Google drive, and work on a copy. If you don't do this, your changes won't be saved!**

In [1]:
# Run this cell to download the test functions for HW 6

import optim as our_optim
import numpy as np

from sequential_expected import (
    test_1_values,
    test_1_sgd_values,
    test_2_values,
    test_2_sgd_values,
)
from test_suite import run_test_suite
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam

from lib import rv
import optim as our_optim
from plotting import tidy_plot, plot_points, plot_fun, plot_heat
from utils import get_data_loader, dataset_paths, run_pytorch

### Optimizers


def f(x):
    return (x - 2.0) * (x - 3.0) * (x + 3.0) * (x + 1.0)


def fdf(x):
    return f(x), 9.0 - (22.0 * x) - (3.0 * x**2) + (4.0 * x**3)


def gd_with_optim(
    f_df,
    x0,
    step_size=0.01,
    step_size_fn=None,
    max_iter=1000,
    optim_cls=our_optim.GD,
    eps=1e-5,
    hook=None,
):
    """
    Runs gradient descent with our custom optimizer class to compute the gradient.
    See below for function for your experiments
    """
    max_iter = min(max_iter, 1000)
    prev_x = x0
    fs = []
    xs = []

    optim = optim_cls(shape=x0.shape)

    for i in range(max_iter):
        prev_f, prev_grad = f_df(prev_x)
        if prev_grad is None:
            prev_grad = num_grad(lambda x: f_df(x)[0])(prev_x)

        fs.append(float(prev_f))
        xs.append(prev_x)

        step = step_size_fn(i) if step_size_fn else step_size

        assert prev_x.shape == prev_grad.shape

        x = prev_x - step * optim.get_grad_step(i + 1, prev_grad)

        if hook:
            hook(x)
        if np.all(abs(x - prev_x) < eps):
            f, _ = f_df(x)
            fs.append(float(f))
            xs.append(x)
            return x, fs, xs
        prev_x = x
    return x, fs, xs


def run_gd_with_optim(
    step_size=0.01,
    init_val=1.0,
    num_steps=10,
    step_size_fn=None,
    optim_cls=our_optim.Adam,
):
    """
    Use this function to run experiments with different optimizers (GD and Adam)
    """
    init_weights = rv([init_val])
    w, js, ws = gd_with_optim(
        fdf,
        init_weights,
        optim_cls=optim_cls,
        step_size=step_size,
        step_size_fn=step_size_fn,
        max_iter=num_steps,
    )
    text_output = "objective", js[-1], "thetas", w
    nax = tidy_plot(
        -4,
        4,
        -25,
        25,
        xlabel="\u03f4",
        ylabel="f(\u03f4)",
        title="step size = " + str(step_size),
        center=True,
    )
    plot_points(
        [float(w) for w in ws],
        [float(j) for j in js],
        nax,
        mark_initial=True,
        mark_final=True,
    )
    plot_fun(nax, lambda w: fdf(w)[0], -4, 4)
    plt.show()
    return text_output


### PyTorch


def run_pytorch_2d(
    data_name,
    layers,
    epochs,
    split=0.25,
    display=True,
    verbose=True,
    trials=1,
    batch_size=32,
    return_history=False,
):
    """
    Trains and evaluates a given model `layers` with the dataset specified by `data_name`
    You should use the `archs` function above to create your model.

    Example usage:

        layers = [nn.Linear(in_features=2, out_features=2, bias=True), nn.ReLU(), nn.Linear(in_features=2, out_features=2, bias=True)]

        # this will train on dataset "1"
        model = run_pytorch_2d("1", layers, epochs=200, trials = 1, verbose=True, display=True)

    #If you don't care about the return values, throw them into a trash variable:

        _ = run_pytorch_2d("2", layers, epochs=200, display=False, verbose=False, trials=5)
    """
    print("Pytorch FC: dataset=", data_name)
    train_dataset_path, val_dataset_path, test_dataset_path = dataset_paths(data_name)
    # Load the datasets
    train_iter, num_classes = get_data_loader(train_dataset_path, batch_size)
    val_iter, num_classes = get_data_loader(val_dataset_path, batch_size)
    test_iter, num_classes = get_data_loader(test_dataset_path, batch_size)

    if val_iter is None:
        # Use split
        print("Use split", train_iter)
        assert split > 0, "`split` must be > 0"
        train_iter, val_iter, num_classes = get_data_loader(
            train_dataset_path, batch_size, split
        )

    val_acc, test_acc = 0, 0
    X_train = torch.cat([batch.X for batch in train_iter], 0)
    y_train = torch.cat([batch.y for batch in train_iter], 0)

    for trial in range(trials):
        trial_history = {
            "epoch_loss": [],
            "epoch_val_loss": [],
            "epoch_acc": [],
            "epoch_val_acc": [],
        }

        if verbose:
            print("\n")
        print(f"# Trial {trial}")

        # Run the model
        (
            model,
            vacc,
            tacc,
        ) = run_pytorch(
            train_iter,
            val_iter,
            test_iter,
            layers,
            epochs,
            split=split,
            verbose=verbose,
            history=trial_history,
        )

        val_acc += vacc if vacc else 0
        test_acc += tacc if tacc else 0
        if display:
            # plot classifier landscape on training data
            plot_heat(X_train, y_train, model)
            plt.title("Training data")
            plt.show()
            if test_iter is not None:
                # plot classifier landscape on testing data
                X_test = torch.cat([batch.X for batch in test_iter], 0)
                y_test = torch.cat([batch.y for batch in test_iter], 0)
                plot_heat(X_test, y_test, model)
                plt.title("Testing data")
                plt.show()
            # Plot epoch loss
            plt.figure(facecolor="white")
            plt.plot(
                range(epochs), trial_history["epoch_loss"], label="epoch_train_loss"
            )
            plt.plot(
                range(epochs), trial_history["epoch_val_loss"], label="epoch_val_loss"
            )
            plt.xlabel("epoch")
            plt.ylabel("loss")
            plt.title("Epoch val_loss and loss")
            plt.legend()
            plt.show()
            # Plot epoch accuracy
            plt.figure(facecolor="white")
            plt.plot(range(epochs), trial_history["epoch_acc"], label="epoch_train_acc")
            plt.plot(
                range(epochs), trial_history["epoch_val_acc"], label="epoch_val_acc"
            )
            plt.xlabel("epoch")
            plt.ylabel("accuracy")
            plt.legend()
            plt.title("Epoch val_acc and acc")
            plt.show()
    if val_acc:
        print("\nAvg. validation accuracy:" + str(val_acc / trials))
    if test_acc:
        print("\nAvg. test accuracy:" + str(test_acc / trials))
    if return_history:
        return model, trial_history
    return model


class Module:
    def sgd_step(self, lrate):
        pass  # For modules w/o weights


### Data sets
# Note that these are not the same as in hw5,
# as y is augmented to look like the outputs from softmax


def for_softmax(y):
    return np.vstack([1 - y, y])


def super_simple_separable_through_origin():
    X = np.array([[2, 3, 9, 12], [5, 1, 6, 5]])
    y = np.array([[1, 0, 1, 0]])
    return X, for_softmax(y)


def super_simple_separable():
    X = np.array([[2, 3, 9, 12], [5, 2, 6, 5]])
    y = np.array([[1, 0, 1, 0]])
    return X, for_softmax(y)


def xor():
    X = np.array([[1, 2, 1, 2], [1, 2, 2, 1]])
    y = np.array([[1, 1, 0, 0]])
    return X, for_softmax(y)


def xor_more():
    X = np.array([[1, 2, 1, 2, 2, 4, 1, 3], [1, 2, 2, 1, 3, 1, 3, 3]])
    y = np.array([[1, 1, 0, 0, 1, 1, 0, 0]])
    return X, for_softmax(y)


def hard():
    X = np.array(
        [
            [
                -0.23390341,
                1.18151883,
                -2.46493986,
                1.55322202,
                1.27621763,
                2.39710997,
                -1.3440304,
                -0.46903436,
                -0.64673502,
                -1.44029872,
                -1.37537243,
                1.05994811,
                -0.93311512,
                1.02735575,
                -0.84138778,
                -2.22585412,
                -0.42591102,
                1.03561105,
                0.91125595,
                -2.26550369,
            ],
            [
                -0.92254932,
                -1.1030963,
                -2.41956036,
                -1.15509002,
                -1.04805327,
                0.08717325,
                0.8184725,
                -0.75171045,
                0.60664705,
                0.80410947,
                -0.11600488,
                1.03747218,
                -0.67210575,
                0.99944446,
                -0.65559838,
                -0.40744784,
                -0.58367642,
                1.0597278,
                -0.95991874,
                -1.41720255,
            ],
        ]
    )
    y = np.array(
        [
            [
                1.0,
                1.0,
                0.0,
                1.0,
                1.0,
                1.0,
                0.0,
                0.0,
                0.0,
                0.0,
                0.0,
                1.0,
                1.0,
                1.0,
                0.0,
                0.0,
                0.0,
                1.0,
                1.0,
                0.0,
            ]
        ]
    )
    return X, for_softmax(y)


### Test cases


def test_dReLU_dz(fn):
    """
    Tests dReLU_dz(): dReLU_dz(dReLU_dz)
    """
    cases = [
        (np.array([[1.0], [0.0], [-5.0]]), np.array([[1.0], [0.0], [0.0]])),
        (
            np.array([[8.7], [-4.0], [-595.006], [0.0], [3.0]]),
            np.array([[1.0], [0.0], [0.0], [0.0], [1.0]]),
        ),
    ]
    run_test_suite(cases, fn)


def test_linear(module):
    """
    Tests Linear module: test_linear(Linear)
    """

    def nn_linear_forward(x):
        np.random.seed(0)
        linear = module(2, 3)
        return linear.forward(x)

    def nn_linear_forward_bias(x):
        np.random.seed(0)
        linear = module(2, 3)
        linear.W0 = np.array([[1], [1], [1]])
        return linear.forward(x)

    def nn_linear_backward(x):
        np.random.seed(0)
        linear = module(2, 3)
        linear.forward(x)
        dLdZ = np.array([[1, 1, 0, 0], [2, 0, 1, 0], [3, 0, 0, 1]])
        dLdA = linear.backward(dLdZ)
        return dLdA, linear.dLdW, linear.dLdW0

    def nn_linear_sgd(x):
        np.random.seed(0)
        linear = module(2, 3)
        linear.forward(x)
        dLdZ = np.array([[1, 1, 0, 0], [2, 0, 1, 0], [3, 0, 0, 1]])
        linear.backward(dLdZ)
        linear.sgd_step(0.005)
        return np.vstack([linear.W, linear.W0.T])

    X, y = super_simple_separable()

    print("***********************************")
    print("Test linear forward:")
    expected = np.array(
        [
            [
                10.417500637754383,
                6.911221682745654,
                20.733665048236965,
                22.891234399772113,
            ],
            [
                7.168722346625092,
                3.489987464919749,
                10.469962394759248,
                9.998261102396512,
            ],
            [
                -2.071054548689073,
                0.6941371647696142,
                2.0824114943088414,
                4.849668106971125,
            ],
        ]
    )
    run_test_suite([(X, expected)], nn_linear_forward)
    print("***********************************")

    print("Test linear forward w bias:")
    run_test_suite([(X, expected + 1)], nn_linear_forward_bias)
    print("***********************************")

    print("Test linear backward output:")
    expected = np.array(
        [
            [
                3.889497924054116,
                1.247373376201773,
                0.2829538755771419,
                0.6920722655660196,
            ],
            [
                2.1525571673658237,
                1.5845507770701677,
                1.3205629190941617,
                -0.6910398159642225,
            ],
        ]
    )
    run_test_suite([(X, expected)], lambda *args: nn_linear_backward(*args)[0])
    print("***********************************")

    print("Test linear backward updated dLdW:")
    expected = np.array([[5, 13, 18], [7, 16, 20]])
    run_test_suite([(X, expected)], lambda *args: nn_linear_backward(*args)[1])
    print("***********************************")

    print("Test linear backward updated dLdW0:")
    expected = np.array([[2], [3], [4]])
    run_test_suite([(X, expected)], lambda *args: nn_linear_backward(*args)[2])
    print("***********************************")

    print("Test linear sgd updated weights:")
    expected = np.array(
        [
            [1.222373376201773, 0.2179538755771419, 0.6020722655660197],
            [1.5495507770701678, 1.2405629190941616, -0.7910398159642225],
            [-0.01, -0.015, -0.02],
        ]
    )
    run_test_suite([(X, expected)], lambda *args: nn_linear_sgd(*args))
    print("***********************************")


def test_sigmoid(module):
    """
    Tests Sigmoid module: test_sigmoid(Sigmoid)
    """

    def nn_sigmoid_forward(x):
        np.random.seed(0)
        sigmoid = module()
        return sigmoid.forward(x)

    def nn_sigmoid_backward(x):
        np.random.seed(0)
        sigmoid = module()
        sigmoid.forward(x)
        dLdA = np.array([[1, 1, 0, 0], [2, 0, 1, 0]])
        return sigmoid.backward(dLdA)

    X, y = super_simple_separable()

    print("***********************************")
    print("Test sigmoid forward:")
    expected_forward = np.array(
        [
            [
                0.88079708,
                0.95257413,
                0.99987661,
                0.99999386,
            ],
            [
                0.99330715,
                0.88079708,
                0.99752738,
                0.99330715,
            ],
        ]
    )
    run_test_suite([(X, expected_forward)], nn_sigmoid_forward)
    print("***********************************")

    print("Test sigmoid backward:")
    expected_backward = np.array(
        [
            [0.10499359, 0.04517666, 0.0, 0.0],
            [0.01329611, 0.0, 0.00246651, 0.0],
        ]
    )
    run_test_suite([(X, expected_backward)], nn_sigmoid_backward)
    print("***********************************")


def test_relu(module):
    """
    Tests ReLU module: test_relu(ReLU)
    """

    def nn_relu_forward(x):
        np.random.seed(0)
        relu = module()
        return relu.forward(x)

    def nn_relu_backward(x):
        np.random.seed(0)
        relu = module()
        relu.forward(x)
        dLdA = np.array([[1, 1, 0, 0], [2, 0, 1, 0]])
        return relu.backward(dLdA)

    X, y = super_simple_separable()

    print("***********************************")
    print("Test relu forward:")
    expected_forward = np.array([[2, 3, 9, 12], [5, 2, 6, 5]])
    run_test_suite([(X, expected_forward)], nn_relu_forward)
    print("***********************************")

    print("Test relu backward:")
    expected_backward = np.array([[1, 1, 0, 0], [2, 0, 1, 0]])
    run_test_suite([(X, expected_backward)], nn_relu_backward)
    print("***********************************")


def test_softmax(module):
    """
    Tests SoftMax module: test_softmax(SoftMax)
    """

    def nn_softmax_forward(x):
        np.random.seed(0)
        softmax = module()
        return softmax.forward(x)

    def nn_softmax_backward(x):
        np.random.seed(0)
        softmax = module()
        Ypred = softmax.forward(x)
        return softmax.class_fun(Ypred)

    X, y = super_simple_separable()

    print("***********************************")
    print("Test softmax forward:")
    expected_forward = np.array(
        [
            [
                0.04742587317756679,
                0.7310585786300048,
                0.9525741268224334,
                0.9990889488055993,
            ],
            [
                0.9525741268224333,
                0.2689414213699951,
                0.04742587317756678,
                0.0009110511944006454,
            ],
        ]
    )
    run_test_suite([(X, expected_forward)], nn_softmax_forward)
    print("***********************************")

    print("Test softmax class fun:")
    expected_backward = np.array([1, 0, 0, 0])
    run_test_suite([(X, expected_backward)], nn_softmax_backward)
    print("***********************************")


def test_nll(module):
    """
    Tests NLL module: test_nll(NLL)
    """
    y = np.array([[1, 0, 1, 0]])
    Y = for_softmax(y)
    ypred = np.array([[0.7, 0.3, 0.99, 0.99]])
    Ypred = for_softmax(ypred)

    def nn_nll_backward(_):
        nll = module()
        nll.forward(Ypred, Y)
        return nll.backward()

    print("***********************************")
    print("Test nll forward:")
    expected_forward = 5.328570409719057
    run_test_suite(
        [
            (
                None,
                expected_forward,
            )
        ],
        lambda _: module().forward(Ypred, Y),
    )
    print("***********************************")

    print("Test nll class fun:")
    expected_backward = np.array(
        [
            [0.30000000000000004, -0.30000000000000004, 0.010000000000000009, -0.99],
            [-0.30000000000000004, 0.3, -0.010000000000000009, 0.99],
        ]
    )
    run_test_suite(
        [
            (
                None,
                expected_backward,
            )
        ],
        nn_nll_backward,
    )
    print("***********************************")


def unit_test(name, expected, actual):
    if actual is None:
        print(name + ": unimplemented")
    elif np.allclose(expected, actual):
        print(name + ": OK")
    else:
        print(name + ": FAILED")
        print("expected: " + str(expected))
        print("but was: " + str(actual))


def test_sequential_components(nn, test_values):
    """Run one step of GD on a simple dataset with the specified
    network, and with batch size (b) = len(dataset), testing the
    behavior of each component of the neural network

    :param nn: A "Sequential" object representing a neural network

    :param test_values: A dictionary containing the expected values
    for the necessary unit tests

    """
    lrate = 0.005
    # data
    X, Y = super_simple_separable()

    # define the modules
    assert len(nn.modules) == 4
    linear_1, f_1, linear_2, f_2 = nn.modules
    Loss = nn.loss

    unit_test("linear_1.W", test_values["linear_1.W"], linear_1.W)
    unit_test("linear_1.W0", test_values["linear_1.W0"], linear_1.W0)
    unit_test("linear_2.W", test_values["linear_2.W"], linear_2.W)
    unit_test("linear_2.W0", test_values["linear_2.W0"], linear_2.W0)

    z_1 = linear_1.forward(X)
    unit_test("z_1", test_values["z_1"], z_1)
    a_1 = f_1.forward(z_1)
    unit_test("a_1", test_values["a_1"], a_1)
    z_2 = linear_2.forward(a_1)
    unit_test("z_2", test_values["z_2"], z_2)
    a_2 = f_2.forward(z_2)
    unit_test("a_2", test_values["a_2"], a_2)

    Ypred = a_2
    loss = Loss.forward(Ypred, Y)
    unit_test("loss", test_values["loss"], loss)
    dloss = Loss.backward()
    unit_test("dloss", test_values["dloss"], dloss)

    dL_dz2 = f_2.backward(dloss)
    unit_test("dL_dz2", test_values["dL_dz2"], dL_dz2)
    dL_da1 = linear_2.backward(dL_dz2)
    unit_test("dL_da1", test_values["dL_da1"], dL_da1)
    dL_dz1 = f_1.backward(dL_da1)
    unit_test("dL_dz1", test_values["dL_dz1"], dL_dz1)
    dL_dX = linear_1.backward(dL_dz1)
    unit_test("dL_dX", test_values["dL_dX"], dL_dX)

    linear_1.sgd_step(lrate)
    unit_test("updated_linear_1.W", test_values["updated_linear_1.W"], linear_1.W)
    unit_test("updated_linear_1.W0", test_values["updated_linear_1.W0"], linear_1.W0)
    linear_2.sgd_step(lrate)
    unit_test("updated_linear_2.W", test_values["updated_linear_2.W"], linear_2.W)
    unit_test("updated_linear_2.W0", test_values["updated_linear_2.W0"], linear_2.W0)


def test_sequential_sgd(nn, test_values):
    """Run one step of SGD on a simple dataset with the specified
    network

    :param nn: A "Sequential" object representing a neural network

    :param test_values: A dictionary containing the expected values
    for the necessary unit tests
    """
    lrate = 0.005
    # data
    X, Y = super_simple_separable()

    # define the modules
    assert len(nn.modules) == 4
    linear_1, f_1, linear_2, f_2 = nn.modules

    unit_test("linear_1.W", test_values["linear_1.W"], linear_1.W)
    unit_test("linear_1.W0", test_values["linear_1.W0"], linear_1.W0)
    unit_test("linear_2.W", test_values["linear_2.W"], linear_2.W)
    unit_test("linear_2.W0", test_values["linear_2.W0"], linear_2.W0)

    nn.sgd(X, Y, iters=1, lrate=lrate)

    unit_test("updated_linear_1.W", test_values["updated_linear_1.W"], linear_1.W)
    unit_test("updated_linear_1.W0", test_values["updated_linear_1.W0"], linear_1.W0)
    unit_test("updated_linear_2.W", test_values["updated_linear_2.W"], linear_2.W)
    unit_test("updated_linear_2.W0", test_values["updated_linear_2.W0"], linear_2.W0)


def test_sequential(sequential, linear, sigmoid, relu, softmax, nll):
    """
    Tests Sequential module: test_sequential(Sequential, Linear, Sigmoid, ReLU, SoftMax, NLL)
    """
    print("***********************************")
    print("Test for sigmoid activation and softmax output:")
    np.random.seed(0)
    test_sequential_components(
        sequential([linear(2, 3), sigmoid(), linear(3, 2), softmax()], nll()),
        test_1_values,
    )
    print("Test sgd:")
    np.random.seed(0)
    test_sequential_sgd(
        sequential([linear(2, 3), sigmoid(), linear(3, 2), softmax()], nll()),
        test_1_sgd_values,
    )
    print("***********************************")

    print("Test for relu activation and softmax output:")
    np.random.seed(0)
    test_sequential_components(
        sequential([linear(2, 3), relu(), linear(3, 2), softmax()], nll()),
        test_2_values,
    )
    print("Test sgd:")
    np.random.seed(0)
    test_sequential_sgd(
        sequential([linear(2, 3), relu(), linear(3, 2), softmax()], nll()),
        test_2_sgd_values,
    )
    print("***********************************")


import numpy as np
import torch
import torch.nn as nn

In [2]:
def derivative_relu(x):
    return np.where(x > 0, 1, 0)

In [3]:
test_dReLU_dz(derivative_relu)

Test case passed!
----------------------------
Test case passed!
----------------------------

All tests passed!


Remember: To look at the functions and test cases we've provided, after running the import cell above, click on the 📁 icon on the sidebar and double-click the `hw06_tests.py` file to open it in a new window. Running a test case should hopefully straightforward: find the corresponding function (usually named `test_<name-of-func>` and pass your implemented function into it).

# 5) Implementing Neural Networks

## 5.1)

lthough for "real" applications you
want to use one of the many packaged implementations of neural networks (we'll start using one of those soon),
there is no substitute for implementing one yourself to get an in-depth understanding. Luckily, that is relatively easy to do if we're not too concerned
with maximum efficiency.
We'll use the modular implementation that we guided you through in the previous problem, which leads to clean code. The basic framework for GD
training is given below. We can construct a network and train it as follows:
```
net = Sequential([Linear(2,3), Tanh(),
Linear(3,3), Tanh(),
Linear(3,2), SoftMax()], NLL())
# train the network on data and labels
net.gd(X, Y)
```
We call this a Sequential object because we feed the outputs of the previous layer into the next, you could say this forms a sequential relationship between layers

We will (later) be generalizing GD to operate on a "mini-batch" of data points instead of a single point. You should strive for an implementation of the forward, backward, and class_fun methods that works with batches of data. Note that when b is mentioned as part of the shape of a matrix in the code, this b refers to the number of data points.


You will need to fill in the missing code to create a sequential neural network model. We encourage you to test in your own Python environment and then paste your answer and verify the results. The test cases are provided in the code distribution or the colab file linked above in this page. The code
distribution includes the tests being run by the code checkers and additional test methods that will test each of the methods in turn, so you
can debug incrementally.


### 5.1) Linear Modules

**Each linear module has a forward method that takes in a batch of activations A  (from the previous layer) and returns a batch of pre-activations Z. Each linear module has a forward method that takes in a batch of activations A
(from the previous layer) and returns a batch of pre-activations Z.Each linear module has a backward method that takes dLdZ and returns dLdA.
This module computes and stores  dLdW and dLdW0, the gradients with respect to the weights.**



Each linear module has a forward method that takes in a column vector of activations A Z
(from the previous layer) and returns a column vector
of pre-activations; it also stores its input or output vectors for use by other methods (e.g., for subsequent backpropagation).
Each activation module has a forward method that takes in a column vector of pre-activations also stores its input or output vectors for use by other methods (e.g., for subsequent backpropagation).
Z A
and returns a column vector of activations; it
Each linear module has a backward method that takes in a column vector and stores ∂Loss
∂W
and ∂Loss
∂W
, the gradients with respect to the weights.
0
∂Loss
∂Z
and returns a column vector ∂Loss
∂A
. This module also computes
Each activation module has a backward method that takes in a column vector ∂Loss
∂A
∂Loss
and returns a column vector .
∂Z
The backpropagation algorithm will consist of:
Calling the forward method of each module in turn, feeding the output of one module as the input to the next; starting with the input values of
the network. After this pass, we have a predicted value for the final network output.
Calling the backward method of each module in reverse order, using the returned value from one module as the input value of the previous one.
The starting value for the backward method is ∂Loss(aL, y)
/∂aL aL
, where is the activation of the final layer (computed during the forward
pass) and y
is the desired output (the label).

In [12]:
class Module:
    def sgd_step(self, lrate):
        pass  # For modules w/o weights


class Linear(Module):
    def __init__(self, m, n):
        # initializes the weights randomly and offsets as 0
        self.m, self.n = (m, n)  # (in size, out size)
        self.W0 = np.zeros([self.n, 1])  # (n x 1)
        self.W = np.random.normal(0, 1.0 * m ** (-0.5), [m, n])  # (m x n)

    def forward(self, A):
        # store the input matrix for future use
        self.A = A  # (m x b)  Hint: make sure you understand what b stands for
        self.Z = self.W.T @ self.A + self.W0  # Your code (n x b)
        return self.Z

    def backward(self, dLdZ):  # dldz = dldout for linear
        # dLdZ is (n x b), uses stored self.A
        # store the derivatives for use in sgd_step and return dLdA
        print(dLdZ)
        self.dLdW = (self.A @ dLdZ.T)
        self.dLdW0 = dLdZ.sum(axis=1).reshape(-1,1)
        return self.W @ dLdZ

    def sgd_step(self, lrate):  # Gradient descent step
        self.W -= lrate * self.dLdW
        self.W0 -= lrate * self.dLdW0

In [13]:
test_linear(Linear)

***********************************
Test linear forward:
Test case passed!
----------------------------

All tests passed!
***********************************
Test linear forward w bias:
Test case passed!
----------------------------

All tests passed!
***********************************
Test linear backward output:
[[1 1 0 0]
 [2 0 1 0]
 [3 0 0 1]]
Test case passed!
----------------------------

All tests passed!
***********************************
Test linear backward updated dLdW:
[[1 1 0 0]
 [2 0 1 0]
 [3 0 0 1]]
Test case passed!
----------------------------

All tests passed!
***********************************
Test linear backward updated dLdW0:
[[1 1 0 0]
 [2 0 1 0]
 [3 0 0 1]]
Test case passed!
----------------------------

All tests passed!
***********************************
Test linear sgd updated weights:
[[1 1 0 0]
 [2 0 1 0]
 [3 0 0 1]]
Test case passed!
----------------------------

All tests passed!
***********************************


In [20]:
x = np.array([[2, 3, 9, 12], [5, 2, 6, 5]]) + np.array([[1, 3]]).T

In [21]:
x.shape

(2, 4)

In [24]:
x.sum(axis=1).reshape(2, 1)

array([[30],
       [30]])

In [25]:
x.sum(axis=1).reshape(-1, 1)

array([[30],
       [30]])

5.2.1) Tanh

In [21]:
class Tanh(Module):  # Layer activation
    def forward(self, Z):
        self.A = np.tanh(Z)
        return self.A

    def backward(self, dLdA):  # Uses stored self.A
        return dLdA * (1-np.tanh(np.arctanh(self.A))**2) # Your code: return dLdZ

In [20]:
test_tanh(Tanh)

NameError: name 'test_tanh' is not defined

5.2.2) ReLU

In [24]:
class ReLU(Module):  # Layer activation
    def forward(self, Z):
        self.A = np.maximum(0, Z)  # Your code
        return self.A

    def backward(self, dLdA):  # uses stored self.A
        return  dLdA * (self.A !=0)# Your code: return dLdZ


In [25]:
test_relu(ReLU)

***********************************
Test relu forward:
Test case passed!
----------------------------

All tests passed!
***********************************
Test relu backward:
Test case passed!
----------------------------

All tests passed!
***********************************


In [28]:
np.maximum([0],[1])

TypeError: 'list' object cannot be interpreted as an integer

5.2.3) Softmax

In [ ]:
class SoftMax(Module):  # Output activation
    def forward(self, Z):
        return None  # Your code

    def backward(self, dLdZ):  # Assume that dLdZ is passed in
        return dLdZ

    def class_fun(self, Ypred):
        # Returns the index of the most likely class for each point as vector of shape (b,)
        return None  # Your code

In [ ]:
test_softmax(SoftMax)

5.3) Loss Module - NLLM

In [ ]:
class NLL(Module):  # Loss
    def forward(self, Ypred, Y):
        # returns loss as a float
        self.Ypred = Ypred
        self.Y = Y
        return None  # Your code

    def backward(self):  # Use stored self.Ypred, self.Y
        # note, this is the derivative of loss with respect to the input of softmax
        return None  # Your code

In [ ]:
test_nll(NLL)

5.4) Sequential

In [ ]:
class Sequential:
    def __init__(self, modules, loss):  # List of modules, loss module
        self.modules = modules
        self.loss = loss

    def sgd(self, X, Y, iters=100, lrate=0.005):  # Train
        D, N = X.shape
        for it in range(iters):
            pass  # YOUR CODE HERE

    def forward(self, Xt):  # Compute Ypred
        for m in self.modules:
            Xt = m.forward(Xt)
        return Xt

    def backward(self, delta):  # Update dLdW and dLdW0
        # Note reversed list of modules
        for m in self.modules[::-1]:
            delta = m.backward(delta)

    def sgd_step(self, lrate):  # Gradient descent step
        for m in self.modules:
            m.sgd_step(lrate)

    def print_accuracy(self, it, X, Y, cur_loss, every=250):
        # Utility method to print accuracy on full dataset, should
        # improve over time when doing SGD. Also prints COURSE/questions/nn loss,
        # which should decrease over time. Call this on each iteration
        # of SGD!
        if it % every == 1:
            cf = self.modules[-1].class_fun
            acc = np.mean(cf(self.forward(X)) == cf(Y))
            print("Iteration =", it, "	Acc =", acc, "	Loss =", cur_loss)

In [ ]:
test_sequential(Sequential, Linear, Tanh, ReLU, SoftMax, NLL)

# 6) Neural Network Packages

**For problem 6, use the function `run_pytorch_2d` in `hw06_tests.py`**

Here's an example to help you get started.

In [ ]:
layers = [
    nn.Linear(in_features=2, out_features=2, bias=True),
    nn.ReLU(),
    nn.Linear(in_features=2, out_features=2, bias=True),
]
_ = run_pytorch_2d(
    "2", layers=layers, epochs=200, trials=1, verbose=False, display=True
)

Here, each layer is an element of the layers list. in_features is the number of units in the previous layer. out_features is the number of units in the current layer. We consider the first layer to be the hidden layer. So you should be able to just modify a parameter of the example we gave.

##6.5)

In [ ]:
layers = [
    nn.Linear(in_features=2, out_features=100, bias=True),
    nn.ReLU(),
    nn.Linear(in_features=100, out_features=100, bias=True),
    nn.ReLU(),
    nn.Linear(in_features=100, out_features=2, bias=True),
]
# By setting `return_history=True` argument in the `run_pytorch_2d` call, we can
# have all the history values returned for inspection
model, history = run_pytorch_2d(
    "2",
    layers=layers,
    epochs=300,
    trials=5,
    verbose=False,
    display=True,
    return_history=True,
)
# the history variable will be a dictionary, with the following keys:
# print(history.keys())
# "epoch_loss", "epoch_val_loss", "epoch_acc", "epoch_val_acc"

## 6.7

You may find the cell below useful for starting your work for this question.

In [ ]:
points = np.array(
    [[-1, 0], [1, 0], [0, -11], [0, 1], [-1, -1], [-1, 1], [1, 1], [1, -1]]
)

deterministic = True
if deterministic:
    torch.manual_seed(10)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(10)

points = torch.Tensor(points)